In [1]:
import httpx
import json

# Wikidata SPARQL Endpoint
WIKIDATA_SPARQL_URL = "https://query.wikidata.org/sparql"

def query_wikidata(qid: str) -> dict:
    # 1. Custom User-Agent (REQUIRED by Wikidata API policy)
    headers = {
        "Accept": "application/sparql-results+json",
        "User-Agent": "OeNBEnrichmentApiTest/1.0 (library-enrichment@example.com)"
    }

    # 2. SPARQL Query
    # Extracts property IDs (e.g. P106, P18) and their resolved human-readable labels
    query = f"""
    SELECT ?prop ?valLabel WHERE {{
      BIND(wd:{qid} AS ?item)
      ?item ?p ?val .
      ?propDirect wikibase:directClaim ?p .
      BIND(STRAFTER(STR(?propDirect), "http://www.wikidata.org/prop/direct/") AS ?prop)
      
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "de,en". }}
    }}
    """

    print(f"📡 Querying Wikidata for Q-ID: {qid}...")
    
    # 3. Synchronous HTTP GET Request
    with httpx.Client() as client:
        response = client.get(
            WIKIDATA_SPARQL_URL,
            params={"query": query},
            headers=headers,
            timeout=10.0
        )
        response.raise_for_status()
        return response.json()


def process_sparql_results(data: dict) -> dict[str, list[str]]:
    """Groups raw SPARQL rows by Property ID (P106, P27, P18, etc.)."""
    raw_claims: dict[str, list[str]] = {}
    bindings = data.get("results", {}).get("bindings", [])

    for row in bindings:
        p_id = row.get("prop", {}).get("value")
        v_label = row.get("valLabel", {}).get("value")

        if p_id and v_label:
            # For P31 (Instance of), extract the Q-ID string if it's a URI
            if p_id == "P31" and "http://www.wikidata.org/entity/" in v_label:
                v_label = v_label.split("/")[-1]
                
            raw_claims.setdefault(p_id, []).append(v_label)

    # Deduplicate values per property key
    return {k: list(dict.fromkeys(v)) for k, v in raw_claims.items()}


if __name__ == "__main__":
    # Test with Immanuel Kant (Q9312)
    target_qid = "Q9312"
    
    raw_json = query_wikidata(target_qid)
    claims = process_sparql_results(raw_json)

    print(f"\n✅ Grouped Claims for {target_qid}:")
    print(json.dumps(claims, indent=2, ensure_ascii=False))

📡 Querying Wikidata for Q-ID: Q9312...

✅ Grouped Claims for Q9312:
{}


In [2]:
import httpx
import json

WIKIDATA_SPARQL_URL = "https://query.wikidata.org/sparql"

def query_wikidata(qid: str) -> dict:
    headers = {
        "Accept": "application/sparql-results+json",
        "User-Agent": "OeNBEnrichmentApiTest/1.0 (library-enrichment@example.com)"
    }

    # Clean SPARQL query that directly captures direct property URIs
    query = f"""
    SELECT ?p ?val ?valLabel WHERE {{
      BIND(wd:{qid} AS ?item)
      ?item ?p ?val .
      FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "de,en". }}
    }}
    """

    print(f"📡 Querying Wikidata for Q-ID: {qid}...")
    
    with httpx.Client() as client:
        response = client.get(
            WIKIDATA_SPARQL_URL,
            params={"query": query},
            headers=headers,
            timeout=15.0
        )
        response.raise_for_status()
        return response.json()


def process_sparql_results(data: dict) -> dict[str, list[str]]:
    raw_claims: dict[str, list[str]] = {}
    bindings = data.get("results", {}).get("bindings", [])

    for row in bindings:
        p_uri = row.get("p", {}).get("value", "")
        # Extract property code (e.g. "P106" from "http://www.wikidata.org/prop/direct/P106")
        p_id = p_uri.split("/")[-1] if "/" in p_uri else None

        # Prefer human-readable label if available, otherwise fall back to raw value
        v_label = row.get("valLabel", {}).get("value") or row.get("val", {}).get("value")

        if p_id and v_label:
            # If P31 (Instance of) or P18 (Image), extract clean ID / URL
            if p_id == "P31" and "http://www.wikidata.org/entity/" in v_label:
                v_label = v_label.split("/")[-1]
                
            raw_claims.setdefault(p_id, []).append(v_label)

    # Deduplicate values per property key
    return {k: list(dict.fromkeys(v)) for k, v in raw_claims.items()}


if __name__ == "__main__":
    target_qid = "Q9312"  # Immanuel Kant
    
    raw_json = query_wikidata(target_qid)
    claims = process_sparql_results(raw_json)

    print(f"\n✅ Grouped Claims for {target_qid}:")
    print(json.dumps(claims, indent=2, ensure_ascii=False))

📡 Querying Wikidata for Q-ID: Q9312...

✅ Grouped Claims for Q9312:
{
  "P1150": [
    "BF 4045 - BF 4046",
    "CF 5000 - CF 5017",
    "CF 5000",
    "BF 4045"
  ],
  "P1153": [
    "55978295100"
  ],
  "P1207": [
    "n93081168"
  ],
  "P1263": [
    "891/000029804"
  ],
  "P1266": [
    "37046"
  ],
  "P1280": [
    "6276707"
  ],
  "P1296": [
    "0035328"
  ],
  "P1309": [
    "vtls000970121"
  ],
  "P1315": [
    "886004"
  ],
  "P1343": [
    "Allgemeine Deutsche Biographie",
    "Brockhaus-Efron",
    "Encyclopædia Britannica (1911)",
    "The Nuttall Encyclopædia",
    "Jüdische Enzyklopädie von Brockhaus und Efron",
    "Enzyklopädisches Wörterbuch Granat",
    "The New Student’s Reference Work",
    "Große Sowjetische Enzyklopädie (1969–1978)",
    "Library of the World's Best Literature",
    "Enzyklopädisches Wörterbuch von Brockhaus-Efron",
    "Meyers Konversations-Lexikon, 4. Auflage (1885–1890)",
    "Pedagogues and Psychologists of the World",
    "Astronomers: A Bio